In [1]:
import pandas as pd
import psycopg2

# PostgreSQL 연결
conn = psycopg2.connect(
    dbname="mydb",
    user="user",
    password="pass",
    host="my_postgres",
    port="5432"
)

# Pandas로 데이터 불러오기
df = pd.read_sql(
    """SELECT index
            , recipe_code
            , recipe_name
            , user_id
            , stars
            , text
            , food_category
            , feature 
       FROM new_review""", conn)
print(df.head())

conn.close()


   index  recipe_code         recipe_name         user_id  stars  \
0      0        14299  Creamy White Chili  u_9iFLIhMa8QaG      5   
1      1        14299  Creamy White Chili  u_Lu6p25tmE77j      5   
2      2        14299  Creamy White Chili  u_s0LwgpZ8Jsqq      5   
3      3        14299  Creamy White Chili  u_fqrybAdYjgjG      0   
4      4        14299  Creamy White Chili  u_XXWKwVhKZD69      0   

                                                text food_category  \
0  I tweaked it a little, removed onions because ...    Soup/Chili   
1  Bush used to have a white chili bean and it ma...    Soup/Chili   
2  I have a very complicated white chicken chili ...    Soup/Chili   
3  In your introduction, you mentioned cream chee...    Soup/Chili   
4  Wonderful! I made this for a &#34;Chili/Stew&#...    Soup/Chili   

                            feature  
0  #creamy, #white, #chili, #hearty  
1  #creamy, #white, #chili, #hearty  
2  #creamy, #white, #chili, #hearty  
3  #creamy, #white

/tmp/ipykernel_1024082/3711286665.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


In [2]:
import torch
print(f"MPS 장치를 지원하도록 build가 되었는가? {torch.backends.mps.is_built()}")
print(f"MPS 장치가 사용 가능한가? {torch.backends.mps.is_available()}")


MPS 장치를 지원하도록 build가 되었는가? False
MPS 장치가 사용 가능한가? False


In [2]:
sum(df.text.isnull())

0

In [2]:
# text preprocessing

def text_process(col):
    col = col.lower()
    # remove meaningless words 
    stoplists = ['and', 'but', 'for', 'nor', 'yet', 'the', 'via', 'per', 'also', 'thus', 'once'
                , 'from', 'with', 'into', 'onto', 'over', 'near', 'like', 'upon', 'such', 'some', 'that'
                , '.', ',', 'our', 'was', 'were', 'did', "didn't", 'now', 'this', 'that']
    for s in stoplists:
        col = col.replace(s, '')
    splited_text = col.split(' ')
    # remove keywords which length is under 2
    splited_text = [k for k in splited_text if len(k) > 2]
    return splited_text

In [3]:
df['keyword_ls'] = df.text.apply(lambda x: text_process(x))

In [25]:
# check the result of preprocessing
print(df.keyword_ls[0])
print(df.keyword_ls[1])
print(df.keyword_ls[2])


['tweaked', 'little', 'removed', 'onions', 'because', 'onion', 'haters', 'house', 'used', 'italian', 'seasoning', 'instead', 'just', 'oregano', 'use', 'paprika/', 'cayenne', 'mix', 'little', 'more', 'than', 'recipe', 'called', 'everything', 'bit', 'more', 'hot', 'chili', 'amazing!', 'easy', 'make', 'everyone', 'absolutely', 'loved', 'will', 'staple', 'meal', 'house']
['bush', 'used', 'have', 'white', 'chili', 'bean', 'made', 'recipe', 'simple', 'i’ve', 'written', 'asked', 'please!', 'bring', 'back']
['have', 'very', 'complicated', 'white', 'chicken', 'chili', 'recipe', 'have', 'made', 'years', 'everyone', 'raves', 'saw', 'recipe', 'thought', 'i’d', 'try', 'easy', 'alternative', 'weeknights', 'husb', 'recipe', 'better!', 'easy', 'delicious!', 'cut', 'back', 'slightly', 'crushed', 'oregano', 'cayenne', 'pep', 'orwise', 'made', 'exactly', 'written']


In [4]:
# make dictionary for each recipe to observe frequent keywords
from itertools import chain
from collections import Counter
category = df.food_category.unique().tolist()
recipe_dict = dict()
for c in category:
    ls = df.loc[df.food_category == c]['keyword_ls'].tolist()
    ls = list(chain(*ls))
    cnt_dict = dict(Counter(ls))
    cnt_dict = dict(sorted(cnt_dict.items(), key=lambda item: item[1], reverse=True))
    recipe_dict[c] = cnt_dict


In [6]:
recipe_dict


{'Soup/Chili': {'recipe': 464,
  'chili': 394,
  'make': 313,
  'made': 311,
  'cream': 253,
  'used': 247,
  'have': 245,
  'chicken': 240,
  'added': 201,
  'very': 188,
  'can': 186,
  'will': 177,
  'beans': 172,
  'great': 169,
  'good': 167,
  'out': 156,
  'time': 152,
  'family': 152,
  'loved': 145,
  'one': 145,
  'more': 143,
  'not': 139,
  'just': 134,
  'again': 129,
  'instead': 128,
  'add': 126,
  'love': 125,
  'soup': 124,
  'little': 123,
  'easy': 120,
  'all': 117,
  'really': 113,
  'only': 112,
  'use': 111,
  'had': 110,
  'would': 109,
  'white': 100,
  'cheese': 100,
  'you': 98,
  'flavor': 94,
  'n&#39;t': 94,
  'because': 90,
  'next': 89,
  'green': 88,
  'making': 88,
  'it&#39;s': 86,
  'husb': 84,
  'pep': 83,
  'everyone': 80,
  'half': 80,
  'delicious': 79,
  'turkey': 75,
  'broth': 72,
  'i&#39;ve': 71,
  'even': 71,
  'times': 70,
  'about': 70,
  'first': 69,
  'always': 67,
  'when': 67,
  'thing': 67,
  'favorite': 66,
  'too': 65,
  'definite

In [5]:
from transformers import AlbertTokenizer, AlbertForSequenceClassification
from transformers import Trainer, TrainingArguments
from datasets import load_dataset


/usr/local/lib/python3.8/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# load model
model_name = "albert-base-v2"
tokenizer = AlbertTokenizer.from_pretrained(model_name)
model = AlbertForSequenceClassification.from_pretrained(model_name, num_labels=5) # I'll compare the inference value with stars feature, thus I set num_labels = 5


Some weights of AlbertForSequenceClassification were not initialized from the model checkpoint at albert-base-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [29]:
# load IMDB dataset of sentiment analysis
dataset = load_dataset("imdb")

dataset 

# Tokenizing
def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)


# Tokenize dataset
train_data = dataset['train'].map(tokenize, batched=True)
test_data = dataset['test'].map(tokenize, batched=True)

# Transform PyTorch tensor 
train_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
test_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])


In [10]:
# Setting train arguments
training_args = TrainingArguments(
    output_dir='./results',
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
)

# Use trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
)

# Train model
trainer.train()


/usr/local/lib/python3.8/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,0.350400,0.334435
2,0.308500,0.436536
3,0.180200,0.581876


TrainOutput(global_step=9375, training_loss=0.2899447135416667, metrics={'train_runtime': 192977.4133, 'train_samples_per_second': 0.389, 'train_steps_per_second': 0.049, 'total_flos': 448222291200000.0, 'train_loss': 0.2899447135416667, 'epoch': 3.0})

In [7]:
# target text 
sample_text = df.text.tolist()

# tokenizing text
inputs = tokenizer(sample_text, return_tensors="pt",
                   truncation=True, padding=True, max_length=128)

# inference
outputs = model(**inputs)
logits = outputs.logits

# result
predicted_label = logits.argmax(dim=1).tolist()

